# 00 — Prepare Candidate List

Takes the raw THINGSplus-filtered export (`results.csv`, image-level: one row per image
that passed the memorability/recognizability/holdability/pleasantness/arousal filters) and:

1. Deduplicates to one row per unique concept, keeping each word's highest-memorability image
   (since `memorability_cr` and `recognizability` are image-level, while properties like
   holdability/pleasantness/arousal are concept-level and constant across a word's images).
2. Converts the raw SUBTLEX frequency count into a Zipf-scale value, so it's on the same
   scale as SUBTLEX-DE's `ZipfSUBTLEX` used later in the German steps.

Output: `unique_candidates_english_zipf.csv`, used as input to `01_english_clip_distance.ipynb`.


In [ ]:
import pandas as pd


In [2]:
RESULTS_PATH = "results.csv"
df = pd.read_csv(RESULTS_PATH)
print(f"Loaded {len(df)} image-level rows, {df['Word'].nunique()} unique words")


Loaded 469 image-level rows, 469 unique words


In [3]:
# ===== Flag likely polysemy confounds =====
# SUBTLEX counts all word senses together, so short common nouns that are also
# frequent verbs/function words (e.g. "can", "watch", "match", "key", "stick") will
# show inflated frequency that doesn't reflect the object-noun sense specifically.
# Not auto-excluded here — review and exclude manually in 01_english_clip_distance.ipynb
# (see the EXCLUDE_WORDS list there).
suspect_threshold = df["ZipfSUBTLEX_en"].quantile(0.97)
suspects = df[df["ZipfSUBTLEX_en"] >= suspect_threshold][["Word", "SUBTLEX freq", "ZipfSUBTLEX_en"]]
print(f"Top {len(suspects)} highest-frequency words (check for polysemy confounds):")
print(suspects.sort_values("ZipfSUBTLEX_en", ascending=False))


Top 15 highest-frequency words (check for polysemy confounds):
      Word  SUBTLEX freq  ZipfSUBTLEX_en
0      can      267620.0        6.719948
1    money       32679.0        5.806699
2    woman       22166.0        5.638117
3    watch       16831.0        5.518540
4     hand       14262.0        5.446610
5    phone       13756.0        5.430922
6     book        9026.0        5.247925
7     hair        7831.0        5.186247
8   handle        5529.0        5.035076
9     ball        5353.0        5.021027
10   paper        5271.0        5.014323
11   stick        4953.0        4.987298
12     bag        4796.0        4.973309
13     box        4577.0        4.953011
14   dress        4447.0        4.940497


In [4]:
df.to_csv("output/unique_candidates_english_zipf.csv", index=False)
print(f"Saved output/unique_candidates_english_zipf.csv with {len(df)} words")


Saved output/unique_candidates_english_zipf.csv with 469 words
